# Laboratorio 2 · Incidencias de pasajeros: extraer, comprobar e iterar

**Objetivo:** extraer datos de mensajes y documentos de una aerolínea ficticia, comparar instrucciones sin y con ejemplos y medir aciertos por campo, tiempo y tokens.

Trabajen en el equipo del proyecto y alternen quién ejecuta y quién revisa. Todos los laboratorios se ejecutan en **Google Colab**, con una cuenta personal de Google. Abrir este notebook e instalar el paquete no requiere cuenta de GitHub.

El producto compartido se guarda en `labs/lab02/`; cada integrante conserva su reflexión y su historial. Descarguen los archivos antes de cerrar Colab: los archivos de la sesión son temporales. La pregunta individual se responde en Canvas al cierre del 23/10; producto, reflexión e historial tienen plazo hasta el **27/10/2026 a las 23:59, hora de Chile**.

**Preparación para revisión docente:** el caso y el código están adaptados; la etiqueta pública `lab02` se habilita después de validar y publicar. Las anotaciones son una referencia propuesta que ustedes deben revisar. Esta extracción registra lo que dice el documento; no decide derechos ni autoriza devoluciones.


## 1. Preparar el entorno

Guarda una copia del notebook en tu Drive y ejecuta la celda de instalación. Después agrega tu clave de Gemini como `GEMINI_API_KEY` o `GOOGLE_API_KEY` en **Secretos** (icono de llave de Colab) y permite que este notebook la lea. Nunca escribas su valor en una celda, captura o conversación.

En la celda de configuración, `MODE` puede ser `real` (consulta el modelo) o `simulated` (respuestas inventadas para practicar sin clave). Se abre en modo simulado; para la entrega cambia a `real` y vuelve a ejecutar. Un fallo de conexión nunca cambia silenciosamente a simulación.

El modelo se elige por configuración con el rol `generation`; no necesitas cambiarlo para empezar. El servidor del curso es un respaldo opcional durante la clase, configurado con `CURSO_API_KEY` y `CURSO_BASE_URL` en Secretos.

Para el **ensayo docente antes de publicar**: sube `curso-ia-lab02.zip` al panel Archivos de la sesión antes de instalar; la celda lo detecta y usa esa copia local. El paquete de revisión se genera con `scripts/build_lab02_review.py`. Para estudiantes se usará el enlace de la etiqueta publicada.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("google.colab") is not None:
    # La etiqueta lab02 se publica después de la revisión docente.
    PACKAGE_REF = "lab02"
    LOCAL_PACKAGE = Path("/content/curso-ia-lab02.zip")
    package_source = str(LOCAL_PACKAGE) if LOCAL_PACKAGE.exists() else (
        f"curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@{PACKAGE_REF}")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        package_source,
    ])
    print("Paquete instalado. Continúa con los secretos y la configuración.")
else:
    print("Fuera de Colab se usa el paquete del entorno de desarrollo.")


In [ ]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from curso_ia import (
    create_model, document_input, load_config, load_dataset, load_json,
    save_experiment, show_document, status, timed_run,
)

# Cambia "simulated" por "real" para consultar el modelo con tu clave de Secretos.
MODE = "simulated"  # @param ["simulated", "real"]
os.environ["CURSO_MODE"] = MODE
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
config = load_config(ROOT)
model_ref = config.models["generation"]
print("Modo:", config.mode, "· Modelo:", model_ref)
display(pd.DataFrame(status(config)))


## 2. Leer los documentos y revisar la referencia

El paquete incluye **50 registros sintéticos**: 48 textos y dos imágenes que reproducen dos de esos textos. Empezaremos con **ocho documentos** (seis textos y dos imágenes): 16 consultas y 40 comparaciones de valores por versión. Los pares texto/imagen sirven para comparar formatos; no son pasajeros independientes.

La **referencia** es el conjunto de valores esperados con el que medimos al modelo. Esta referencia fue propuesta por el asistente: revisa los originales y sus notas antes de marcar `REFERENCE_REVIEWED = True`. Si no estás de acuerdo, corrige la referencia en la copia de trabajo antes de consultar el modelo y explica el motivo en `REFERENCE_NOTES`; nunca la cambies solo para que coincida con la respuesta del modelo.

Extraemos cinco valores: reserva afectada, acción solicitada, importe pagado por el pasaje en pesos chilenos, fecha de vuelo y retraso confirmado en minutos. Un dato desconocido o ambiguo se conserva como `None`. Un cero explícito se conserva como 0. La política completa de anotación está en la instrucción de la sección 4.

`USE_FULL_DATASET = True` ejecuta los 50 registros: 100 consultas antes de cualquier reintento. Hazlo solo después de revisar tu cuota y el tiempo disponible. La muestra completa contiene patrones repetidos y no sustituye el conjunto de evaluación independiente del Hito 1.


In [ ]:
DATASET_PATH = None  # El paquete trae los datos; no necesitas subir archivos.
USE_FULL_DATASET = False

dataset = load_dataset(2, DATASET_PATH)
core_ids = set(dataset["core_ids"])
cases = [case for case in dataset["cases"] if USE_FULL_DATASET or case["id"] in core_ids]
REFERENCE_REVIEWED = False
REFERENCE_NOTES = ""  # Anota aquí correcciones o desacuerdos con la referencia propuesta.

print("Conjunto:", dataset["name"], "· Documentos:", len(cases))
display(pd.DataFrame([{"case_id": case["id"], **case["reference"], "note": case["annotation_note"]}
                      for case in cases]))


In [ ]:
for case in cases:
    print("─────", case["id"], "·", case["document"])
    show_document(case["path"])

## 3. Definir el formulario que debe llenar el modelo

El **esquema** es el formulario: dice qué campos debe devolver el modelo y de qué tipo es cada uno. PydanticAI se lo entrega al modelo y revisa que la respuesta lo cumpla; si no lo cumple, le pide que lo corrija una vez.

Cada campo tiene tres partes:

- `value`: el dato extraído. `str | None` significa «texto o vacío»; `int | None`, «número entero o vacío». `Literal[...]` limita las respuestas a las opciones indicadas.
- `confidence`: qué tan seguro *dice* estar el modelo, de 0 a 1. Es una declaración, no una garantía; más adelante la comparamos con los aciertos.
- `evidence`: el fragmento del documento que respalda el valor, o vacío si no hay respaldo.

Las descripciones (`description`) también las lee el modelo: son parte de la instrucción.

In [ ]:
class BookingCode(BaseModel):
    value: str | None = Field(description="Reserva afectada; null si falta o hay varias sin resolver")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento literal del documento que respalda el valor")


class RequestedAction(BaseModel):
    value: Literal["rebooking", "refund"] | None = Field(
        description="Cambio de vuelo o devolución pedidos; null si no hay una sola acción resuelta")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento literal del documento que respalda el valor")


class AmountClp(BaseModel):
    value: int | None = Field(description="Importe pagado por el pasaje en CLP; null si falta o no está en CLP")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento literal del documento que respalda el valor")


class FlightDate(BaseModel):
    value: str | None = Field(description="Fecha completa del vuelo en AAAA-MM-DD; null si no se resuelve")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento literal del documento que respalda el valor")


class DelayMinutes(BaseModel):
    value: int | None = Field(description="Retraso confirmado en minutos enteros; null si es aproximado o falta")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento literal del documento que respalda el valor")


class Extraction(BaseModel):
    booking_code: BookingCode
    requested_action: RequestedAction
    amount_clp: AmountClp
    flight_date: FlightDate
    delay_minutes: DelayMinutes


FIELDS = list(Extraction.model_fields)
print("Campos:", FIELDS)


## 4. Escribir dos instrucciones: sin ejemplos y con ejemplos

Las dos versiones comparten esquema, reglas, modelo y documentos. La única diferencia es que la segunda (*few-shot*) agrega dos ejemplos resueltos al final. Así, si cambian los resultados, sabemos a qué atribuirlo.

Los ejemplos no son documentos de la muestra: si lo fueran, el modelo vería la respuesta antes de ser evaluado.

In [ ]:
BASE_INSTRUCTION = """
Extrae los cinco campos del documento de una aerolínea ficticia. Usa solo el documento.
booking_code: código literal de la reserva afectada; null si falta o hay varias sin resolver.
requested_action: rebooking (cambio de vuelo), refund (devolución) o null.
No deduzcas una solicitud por el daño o la cancelación. Si hay dos alternativas sin decidir, usa null.
amount_clp: importe pagado por el pasaje en pesos chilenos, como entero; no el reembolso solicitado,
ni el equipaje adicional. No sumes impuestos que ya estén incluidos. No conviertas monedas.
Conserva el 0 explícito; usa null si falta, es ilegible, ambiguo o solo está en otra moneda.
flight_date: fecha del vuelo en AAAA-MM-DD. Normaliza fechas completas con formato resuelto.
No completes fechas parciales ni relativas sin ancla; si hay contradicción sin resolver, usa null.
delay_minutes: duración exacta y confirmada del retraso en minutos enteros.
Normaliza horas y minutos expresos; no calcules desde horarios ni confundas duración de vuelo con retraso.
Si el retraso es aproximado, un intervalo, no confirmado o ausente, usa null.
Una corrección explícita del pasajero reemplaza su dato anterior. Si no resuelve el conflicto, usa null.
Devuelve cada campo con value, confidence entre 0 y 1 y evidence.
En evidence copia un fragmento literal breve que respalde el valor; si no hay respaldo, usa null.
No decidas derechos, elegibilidad de reembolso ni acciones sobre reservas.
""".strip()

examples = "\n\n".join(
    f"Documento: {example['text']}\nValores esperados: {json.dumps(example['reference'], ensure_ascii=False)}"
    for example in dataset["prompt_examples"]
)

INSTRUCTIONS = {
    "baseline": BASE_INSTRUCTION,
    "few_shot": BASE_INSTRUCTION + "\n\nEjemplos de valores (devuelve el esquema completo):\n\n" + examples,
}
SETTINGS = {"temperature": 0.0, "max_tokens": 2400}
print(INSTRUCTIONS["few_shot"])


## 5. Ejecutar con PydanticAI

Para cada documento se crea un `Agent` con el modelo, el esquema (`output_type`) y la instrucción, y se le envía el documento. La imagen se envía como imagen, no como su texto.

`timed_run` hace la consulta y anota la respuesta, el tiempo y los tokens. Si una consulta falla, queda registrada como error y cuenta como fallo en las métricas; el resto sigue.

Las dos versiones se alternan documento por documento (sin ejemplos, con ejemplos, siguiente documento…). Si ejecutáramos primero todas las de una versión, cualquier lentitud pasajera del servicio la perjudicaría solo a ella.

En modo simulado las respuestas están preparadas y no se miden tiempos ni tokens.

In [ ]:
rows = []
for case in cases:
    document = document_input(case["path"])
    for variant, instruction in INSTRUCTIONS.items():
        # En modo simulado se usa la respuesta preparada; en modo real se ignora.
        prepared = case.get("prepared_outputs", {}).get(variant)
        model = create_model(model_ref, config, simulated_output=prepared)
        agent = Agent(model, output_type=Extraction, instructions=instruction,
                      model_settings=SETTINGS, retries=1)
        row = await timed_run(agent, ["Extrae los campos del documento adjunto.", document],
                              ref=model_ref, config=config)
        rows.append({"variant": variant, "case_id": case["id"], **row})
        print(case["id"], variant, row["status"], row.get("error_type", ""))

## 6. Guardar la evidencia

Guardamos instrucciones, esquema, configuración sin claves, referencia y todas las respuestas. Copiamos también los documentos usados para que otra persona pueda verificar las citas y recalcular las métricas sin instalar el paquete ni consultar al modelo.

Cada ejecución sobrescribe `experiment.json`. **Descarga la evidencia antes de otra iteración** y conserva cada archivo con un nombre distinto en tu equipo. Guardar una copia del notebook en Drive no guarda los archivos temporales de la sesión.


In [ ]:
evidence_path = ROOT / "labs/lab02/evidence/experiment.json"
source_dir = evidence_path.parent.parent / "documents"
source_dir.mkdir(parents=True, exist_ok=True)
for case in cases:
    (source_dir / case["path"].name).write_bytes(case["path"].read_bytes())

payload = {
    "dataset": dataset["name"],
    "dataset_metadata": dataset["metadata"],
    "models": {"generation": model_ref},
    "settings": SETTINGS,
    "output_schema": Extraction.model_json_schema(),
    "instructions": INSTRUCTIONS,
    "cases": [{key: case[key] for key in ("id", "document", "reference", "annotation_note", "source_id")}
              for case in cases],
    "reference_reviewed": REFERENCE_REVIEWED,
    "reference_notes": REFERENCE_NOTES,
    "results": rows,
    "decisions": {"change": "", "evidence": "", "remaining_error": ""},
}
save_experiment(evidence_path, payload, config)
print("Evidencia guardada en", evidence_path)


## 7. Medir aciertos por campo

Desde aquí se trabaja solo con el archivo guardado.

- **Exactitud**: cuántas veces el valor coincide con la referencia, dividido por todos los documentos. Un vacío frente a una referencia vacía cuenta como acierto. Una consulta fallida cuenta como error.
- **Completitud**: cuántas veces el modelo llenó el campo. Llenar más no es mejor si lo llenado es inventado.

La tabla de errores sirve para ir al documento original y entender qué pasó. La métrica no revisa si la cita de `evidence` es fiel: eso se mira a mano.

In [ ]:
saved = load_json(evidence_path)
reference = {case["id"]: case["reference"] for case in saved["cases"]}

details = []
for row in saved["results"]:
    ok = row["status"] == "ok"
    for field in FIELDS:
        extracted = row["output"][field] if ok else {}
        predicted = extracted.get("value")
        expected = reference[row["case_id"]][field]
        details.append({
            "variant": row["variant"], "case_id": row["case_id"], "field": field,
            "expected": expected, "predicted": predicted,
            "correct": ok and predicted == expected,
            "completed": ok and predicted is not None,
            "failed": not ok,
            "confidence": extracted.get("confidence"),
            "evidence": extracted.get("evidence"),
        })
detail = pd.DataFrame(details)

metrics = detail.groupby(["variant", "field"]).agg(
    correct=("correct", "sum"), cases=("correct", "size"),
    completed=("completed", "sum"), failed=("failed", "sum"),
)
metrics["accuracy"] = metrics["correct"] / metrics["cases"]
metrics["completeness"] = metrics["completed"] / metrics["cases"]
display(metrics)

print("Errores para revisar en el original:")
display(detail[~detail["correct"]][["variant", "case_id", "field", "expected", "predicted", "failed"]])

## 8. Comparar tiempos y tokens

Agregar ejemplos alarga la instrucción: el modelo recibe más **tokens de entrada** (los fragmentos de texto en que se divide lo que lee) en cada consulta. Eso tiene un costo, que medimos junto con el tiempo de respuesta.

El tiempo depende también de la red, de la carga del servicio y de los reintentos automáticos cuando el nivel gratuito limita las solicitudes por minuto. Por eso usamos la **mediana** (el valor del medio, que no se deja arrastrar por una consulta muy lenta) y miramos la tabla por documento. Con ocho documentos y una sola ejecución, una diferencia pequeña no es concluyente.


In [ ]:
runs = pd.DataFrame(saved["results"])
if saved["mode"] == "simulated":
    print("El modo simulado no mide tiempos ni tokens: ejecuta en modo real para esta comparación.")
else:
    successful = runs[runs["status"] == "ok"].copy()
    print("Consultas fallidas:", len(runs) - len(successful))
    if successful.empty:
        print("Todas las consultas fallaron; revisa el bloqueo y conserva la evidencia.")
    else:
        successful["latency_s"] = successful["latency_ms"] / 1000
        timing = successful.groupby("variant").agg(
            median_latency_s=("latency_s", "median"),
            max_latency_s=("latency_s", "max"),
            mean_input_tokens=("input_tokens", "mean"),
            mean_output_tokens=("output_tokens", "mean"),
            total_cost_usd=("estimated_cost_usd", lambda values: values.sum(min_count=1)),
        )
        display(timing.round(2))
        display(successful.pivot(index="case_id", columns="variant", values="latency_s").round(1))


## 9. Contrastar la confianza y explicar el cambio

La primera tabla toma los campos en que el modelo dijo tener al menos 0,8 de confianza y cuenta cuántos acertó. Si declara mucha confianza y se equivoca seguido, no conviene usar esa cifra para decidir sin revisión humana. En modo simulado, las confianzas son inventadas.

Después:

1. Elige un error, búscalo en el documento original y explica por qué ocurrió.
2. Cambia **una sola** parte de la instrucción para corregirlo.
3. Descarga la evidencia actual antes de sobrescribirla y vuelve a ejecutar desde la sección 5. Revisa también si el arreglo rompió algo que antes funcionaba.
4. Completa `DECISIONS` con lo que cambiaste, la evidencia que lo justifica (aciertos, tiempos o tokens) y un error que sigue sin resolverse.
Si no aparece ningún error real, informa ese resultado. Revisa casos difíciles de la ampliación sin retirar aciertos del reporte original; no inventes errores ni presentes las salidas simuladas como mediciones.


In [ ]:
HIGH_CONFIDENCE = 0.8
confident = detail[detail["confidence"] >= HIGH_CONFIDENCE]
if confident.empty:
    print("Ningún campo supera la confianza elegida.")
else:
    confidence_check = confident.groupby("variant").agg(
        fields=("correct", "size"), correct=("correct", "sum"))
    confidence_check["accuracy"] = confidence_check["correct"] / confidence_check["fields"]
    display(confidence_check)

DECISIONS = {
    "change": "",           # Qué cambiaste en la instrucción
    "evidence": "",         # Qué resultado lo justifica
    "remaining_error": "",  # Un error que sigue y por qué
}
saved["decisions"] = DECISIONS
saved["reference_reviewed"] = REFERENCE_REVIEWED
saved["reference_notes"] = REFERENCE_NOTES
save_experiment(evidence_path, saved, config)


## 10. Descargar y entregar

Ejecuta la celda siguiente para descargar un ZIP con la evidencia y los documentos del equipo. Guarda también el notebook **con sus salidas** mediante **Archivo → Descargar → Descargar .ipynb**. Descarga una versión antes de cada iteración que quieras conservar.

**Producto del equipo:** notebook, documentos, `experiment.json` y explicación del cambio en `DECISIONS`. Después se incorporan a `labs/lab02/` del repositorio del equipo; solo lo validado se promueve a `src/`. Para esa incorporación puedes usar el navegador de GitHub cuando tu cuenta y tu repositorio estén habilitados; no hace falta usar Git desde Colab.

**Cada integrante:** reflexión propia, historial completo del asistente usado o declaración de que no usó uno, y enlace al producto compartido. Retira credenciales y datos sensibles e indica las omisiones. La descarga no crea tu reflexión ni el historial por ti.

**Canvas:** responde el cuestionario individual de cinco minutos al cierre de la clase del 23/10. Producto, reflexión e historial se entregan según la tarea del laboratorio hasta el **martes 27/10 a las 23:59, hora de Chile**. Conserva el ZIP mientras el docente habilita los repositorios; abrir y ejecutar este laboratorio no depende de tener una cuenta de GitHub.

**Antes de entregar:** ejecuta en modo real, revisa la referencia, comprueba las citas en los originales y completa las decisiones. La simulación solo comprueba el circuito. Si el proveedor falla, conserva el bloqueo registrado.

Referencias opcionales (inglés, unos 10 minutos cada una): [PydanticAI: salida estructurada](https://pydantic.dev/docs/ai/core-concepts/output/) y [entrada multimodal](https://pydantic.dev/docs/ai/core-concepts/input/), revisadas el 22/10/2026; [Colab: archivos temporales y límites](https://research.google.com/colaboratory/faq.html), inglés, unos 5 minutos, revisada el 22/10/2026.


In [ ]:
import shutil
from datetime import datetime, timezone

# Un nombre distinto conserva las iteraciones descargadas antes de sobrescribirlas.
archive_name = "lab02-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
archive_path = Path(shutil.make_archive(str(ROOT / archive_name), "zip",
                                       root_dir=ROOT / "labs", base_dir="lab02"))
print("Descarga preparada:", archive_path.name)
try:
    from google.colab import files
except ImportError:
    print("Fuera de Colab, copia el ZIP desde la ruta indicada.")
else:
    files.download(str(archive_path))
